# Credit Card Fraud Detection — Step 6: Cost-Sensitive Loss Function & Dollar-Weighted Optimization from Scratch
### Milestone 6 of the 30-Day Capstone Upgrade
**Course:** Python for Data Science (PDS)
**Focus:** Mathematical Formulation of Amount-Weighted Binary Cross-Entropy Loss, Analytical Chain-Rule Gradient Derivations, and Dollar-Weighted Value Recall in Pure NumPy.
**Constraint:** Strictly zero `sklearn` or external ML packages — 100% pure NumPy, Pandas, and Matplotlib.

---

## 1. Theoretical Foundations & Mathematical Formulations

### 1.1 The Theoretical Flaw of Unweighted Binary Cross-Entropy
Standard classification algorithms minimize symmetric empirical risk, where every observation $i$ carries an identical unit weight ($1.0$):
$$\mathcal{L}(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \ln \hat{y}^{(i)} + (1 - y^{(i)}) \ln(1 - \hat{y}^{(i)}) \right]$$

Under this unweighted objective:
- Failing to detect a **\$5,000.00 fraud** incurs the exact same penalty as missing a **\$1.50 test charge**.
- The model allocates equal optimization effort to micro-transactions as it does to catastrophic institutional capital draining.

### 1.2 Mathematical Formulation of the Cost-Sensitive Loss Function
We formulate an **Amount-Weighted Cost-Sensitive Loss Function** (Elkan, 2001; Ting, 2002). For each transaction $i$, we define a financial loss weight $c^{(i)}$:
$$c^{(i)} = \begin{cases}
  1.0 + \kappa \cdot \ln(\text{Amount}^{(i)} + 1), & \text{if } y^{(i)} = 1 \text{ (Fraud)} \\
  1.0, & \text{if } y^{(i)} = 0 \text{ (Legitimate)}
\end{cases}$$
where $\kappa \ge 0$ is the financial sensitivity scaling parameter.

The total cost-sensitive objective to be minimized is:
$$\mathcal{L}_{\text{cost}}(\mathbf{w}, b) = -\frac{1}{\sum_{i=1}^m c^{(i)}} \sum_{i=1}^m c^{(i)} \left[ y^{(i)} \ln(\hat{y}^{(i)}) + (1 - y^{(i)}) \ln(1 - \hat{y}^{(i)}) \right] + \frac{\lambda}{2m} \|\mathbf{w}\|_2^2$$

### 1.3 Derivation of Analytical Cost-Weighted Gradients via Chain Rule
Let $\hat{y}^{(i)} = \sigma(z^{(i)}) = \frac{1}{1 + e^{-z^{(i)}}}$, where $z^{(i)} = \mathbf{w}^T \mathbf{x}^{(i)} + b$.

Applying the multivariate chain rule for sample $i$:
$$\frac{\partial \mathcal{L}_{\text{cost}}^{(i)}}{\partial w_j} = \frac{\partial \mathcal{L}_{\text{cost}}^{(i)}}{\partial \hat{y}^{(i)}} \cdot \frac{\partial \hat{y}^{(i)}}{\partial z^{(i)}} \cdot \frac{\partial z^{(i)}}{\partial w_j}$$

Computing intermediate terms:
$$\frac{\partial \mathcal{L}_{\text{cost}}^{(i)}}{\partial \hat{y}^{(i)}} = -c^{(i)} \left[ \frac{y^{(i)}}{\hat{y}^{(i)}} - \frac{1 - y^{(i)}}{1 - \hat{y}^{(i)}} \right] = c^{(i)} \frac{\hat{y}^{(i)} - y^{(i)}}{\hat{y}^{(i)}(1 - \hat{y}^{(i)})}$$
$$\frac{\partial \hat{y}^{(i)}}{\partial z^{(i)}} = \hat{y}^{(i)}(1 - \hat{y}^{(i)}), \qquad \frac{\partial z^{(i)}}{\partial w_j} = x_j^{(i)}, \qquad \frac{\partial z^{(i)}}{\partial b} = 1$$

Multiplying together yields an elegant, direct cost-scaling cancellation:
$$\frac{\partial \mathcal{L}_{\text{cost}}^{(i)}}{\partial w_j} = c^{(i)} (\hat{y}^{(i)} - y^{(i)}) x_j^{(i)}$$

Summing over all $m$ training samples and incorporating matrix notation (where $\odot$ represents the Hadamard element-wise product):
$$\nabla_{\mathbf{w}} \mathcal{L}_{\text{cost}} = \frac{1}{\sum_{i=1}^m c^{(i)}} \mathbf{X}^T \left( \mathbf{c} \odot (\hat{\mathbf{y}} - \mathbf{y}) \right) + \frac{\lambda}{m} \mathbf{w}$$
$$\frac{\partial \mathcal{L}_{\text{cost}}}{\partial b} = \frac{1}{\sum_{i=1}^m c^{(i)}} \sum_{i=1}^m c^{(i)} (\hat{y}^{(i)} - y^{(i)})$$

### 1.4 Dollar-Weighted Value Recall ($\text{Recall}_{\$}$)
In addition to standard count-based Recall, we introduce **Dollar-Weighted Value Recall**:
$$\text{Recall}_{\$} = \frac{\sum_{i \in \text{TP}} \text{Amount}^{(i)}}{\sum_{i \in \text{Fraud}} \text{Amount}^{(i)}}$$
This measures the exact percentage of total stolen currency successfully protected by the classifier.
---

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print("Environment initialized for Step 6: Cost-Sensitive Loss from Scratch.")

## 2. Ingesting Data & Preparing Features

In [ ]:
possible_paths = [
    os.path.join('..', 'data', 'creditcard.csv'),
    os.path.join('data', 'creditcard.csv'),
    r'C:\Users\HP\Desktop\PDS\data\creditcard.csv',
    r'C:\Users\HP\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\5518A31AAB1A89BF99B5535DB1EFC4DABE33AAC0\transfers\2026-42\archive\creditcard.csv'
]

DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if DATA_PATH and os.path.exists(DATA_PATH):
    print(f"Loading official Kaggle dataset from: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("Dataset not found; generating fallback sample dataset.")
    n_samples = 15000
    n_fraud = int(0.002 * n_samples)
    n_legit = n_samples - n_fraud
    time_feat = np.sort(np.random.uniform(0, 172800, n_samples))
    legit_features = np.random.normal(loc=0.0, scale=1.0, size=(n_legit, 28))
    fraud_features = np.random.normal(loc=1.5, scale=1.8, size=(n_fraud, 28))
    features = np.vstack([legit_features, fraud_features])
    legit_amounts = np.random.exponential(scale=85.0, size=n_legit)
    fraud_amounts = np.random.exponential(scale=140.0, size=n_fraud)
    amounts = np.concatenate([legit_amounts, fraud_amounts])
    labels = np.array([0] * n_legit + [1] * n_fraud)
    col_names = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
    data_mat = np.column_stack([time_feat, features, amounts, labels])
    shuffle_idx = np.random.permutation(n_samples)
    df = pd.DataFrame(data_mat[shuffle_idx], columns=col_names)
    df['Class'] = df['Class'].astype(int)

# Cyclic Time and Log Amount Features
hours = (df['Time'] / 3600.0) % 24.0
df['Time_sin'] = np.sin(2.0 * np.pi * hours / 24.0)
df['Time_cos'] = np.cos(2.0 * np.pi * hours / 24.0)
df['Amount_log'] = np.log1p(df['Amount'])

feature_cols = [f'V{i}' for i in range(1, 29)] + ['Time_sin', 'Time_cos', 'Amount_log']
X_all = df[feature_cols].to_numpy(dtype=np.float64)
y_all = df['Class'].to_numpy(dtype=np.int64)
amounts_all = df['Amount'].to_numpy(dtype=np.float64)

print(f"Total transactions: {len(X_all):,}")
print(f"Total fraud dollar volume: ${amounts_all[y_all == 1].sum():,.2f}")

In [ ]:
# Stratified 80/20 train-test split
np.random.seed(SEED)
idx_0 = np.where(y_all == 0)[0]
idx_1 = np.where(y_all == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)
n_te_0 = int(len(idx_0) * 0.2)
n_te_1 = int(len(idx_1) * 0.2)
test_idx = np.concatenate([idx_0[:n_te_0], idx_1[:n_te_1]])
train_idx = np.concatenate([idx_0[n_te_0:], idx_1[n_te_1:]])
np.random.shuffle(test_idx)
np.random.shuffle(train_idx)

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]
amounts_train, amounts_test = amounts_all[train_idx], amounts_all[test_idx]

# Vectorized SMOTE from Step 1
def smote_scratch(X_min, n_synthetic, k=5, seed=42):
    np.random.seed(seed)
    n_min, n_features = X_min.shape
    k = min(k, n_min - 1)
    dot_prods = np.dot(X_min, X_min.T)
    sq_norms = np.diag(dot_prods)
    dist_matrix_sq = sq_norms[:, np.newaxis] + sq_norms[np.newaxis, :] - 2.0 * dot_prods
    np.maximum(dist_matrix_sq, 0.0, out=dist_matrix_sq)
    np.fill_diagonal(dist_matrix_sq, np.inf)
    knn_indices = np.argpartition(dist_matrix_sq, kth=k, axis=1)[:, :k]
    base_indices = np.random.choice(n_min, size=n_synthetic, replace=True)
    neighbor_col_picks = np.random.randint(0, k, size=n_synthetic)
    chosen_neighbors = knn_indices[base_indices, neighbor_col_picks]
    lambdas = np.random.uniform(0.0, 1.0, size=(n_synthetic, 1))
    diffs = X_min[chosen_neighbors] - X_min[base_indices]
    return X_min[base_indices] + lambdas * diffs

X_min_tr = X_train[y_train == 1]
amounts_min_tr = amounts_train[y_train == 1]
n_syn = len(X_min_tr) * 3
X_syn = smote_scratch(X_min_tr, n_synthetic=n_syn, k=5, seed=SEED)
# Assign synthetic amounts sampled from minority empirical distribution
amounts_syn = np.random.choice(amounts_min_tr, size=n_syn, replace=True)

X_fraud_aug = np.vstack([X_min_tr, X_syn])
amounts_fraud_aug = np.concatenate([amounts_min_tr, amounts_syn])

idx_0_tr = np.where(y_train == 0)[0]
sampled_0 = np.random.choice(idx_0_tr, size=len(X_fraud_aug), replace=False)
X_bal = np.vstack([X_fraud_aug, X_train[sampled_0]])
y_bal = np.concatenate([np.ones(len(X_fraud_aug), dtype=np.int64), np.zeros(len(sampled_0), dtype=np.int64)])
amounts_bal = np.concatenate([amounts_fraud_aug, amounts_train[sampled_0]])

perm = np.random.permutation(len(y_bal))
X_bal, y_bal, amounts_bal = X_bal[perm], y_bal[perm], amounts_bal[perm]

# Standardize
mean = np.mean(X_bal, axis=0)
std = np.std(X_bal, axis=0)
std[std == 0.0] = 1.0
X_tr_norm = (X_bal - mean) / std
X_te_norm = (X_test - mean) / std

print(f"Training set ready: {len(X_tr_norm):,} samples (Balanced)")
print(f"Testing set ready : {len(X_te_norm):,} samples (Held-Out Test Distribution: ${amounts_test[y_test==1].sum():,.2f} fraud)")

## 3. Cost-Sensitive Logistic Regression Implementation from Scratch
Implements analytical cost-scaled gradient updates and loss tracking in pure NumPy.

In [ ]:
class CostSensitiveLogisticRegressionScratch:
    """
    First-principles Logistic Regression minimizing Amount-Weighted Cost-Sensitive Loss.
    c_i = 1.0 + kappa * ln(Amount_i + 1) for fraud, 1.0 for legitimate.
    """
    def __init__(self, lr=0.1, n_iter=1200, l2_reg=0.01, kappa=1.5):
        self.lr = lr
        self.n_iter = n_iter
        self.l2_reg = l2_reg
        self.kappa = kappa
        self.w = None
        self.b = 0.0
        self.loss_history = []
        
    def compute_weights(self, y, amounts):
        """c_i = 1 + kappa * ln(amount + 1) for fraud, 1.0 for legit"""
        log_amt = np.log1p(amounts)
        weights = np.ones_like(y, dtype=np.float64)
        weights[y == 1] += self.kappa * log_amt[y == 1]
        return weights
        
    def fit(self, X, y, amounts):
        m, d = X.shape
        self.w = np.zeros(d, dtype=np.float64)
        self.b = 0.0
        self.loss_history = []
        
        # Compute sample cost weights
        c = self.compute_weights(y, amounts)
        c_sum = np.sum(c)
        
        for _ in range(self.n_iter):
            z = np.clip(np.dot(X, self.w) + self.b, -500.0, 500.0)
            y_hat = 1.0 / (1.0 + np.exp(-z))
            y_hat_clip = np.clip(y_hat, 1e-15, 1.0 - 1e-15)
            
            # Cost-sensitive loss
            bce_i = -(y * np.log(y_hat_clip) + (1.0 - y) * np.log(1.0 - y_hat_clip))
            weighted_loss = np.sum(c * bce_i) / c_sum + (self.l2_reg / (2.0 * m)) * np.sum(self.w ** 2)
            self.loss_history.append(weighted_loss)
            
            # Analytical Cost-Weighted Gradients
            err = c * (y_hat - y)
            dw = (1.0 / c_sum) * np.dot(X.T, err) + (self.l2_reg / m) * self.w
            db = (1.0 / c_sum) * np.sum(err)
            
            self.w -= self.lr * dw
            self.b -= self.lr * db
            
        return self
        
    def predict_proba(self, X):
        z = np.clip(np.dot(X, self.w) + self.b, -500.0, 500.0)
        return 1.0 / (1.0 + np.exp(-z))
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print("CostSensitiveLogisticRegressionScratch defined successfully in pure NumPy.")

## 4. Head-to-Head Training: Standard vs. Cost-Sensitive Model

In [ ]:
# 1. Standard Unweighted Model (kappa = 0.0)
standard_model = CostSensitiveLogisticRegressionScratch(lr=0.1, n_iter=1200, l2_reg=0.01, kappa=0.0)
standard_model.fit(X_tr_norm, y_bal, amounts_bal)

# 2. Cost-Sensitive Model (kappa = 2.0)
cost_model = CostSensitiveLogisticRegressionScratch(lr=0.1, n_iter=1200, l2_reg=0.01, kappa=2.0)
cost_model.fit(X_tr_norm, y_bal, amounts_bal)

print("Both models trained successfully!")

## 5. Comparative Evaluation on Held-Out Test Set
We compare standard count-based metrics vs. **Dollar-Weighted Value Recall**.

In [ ]:
def trapezoid_scratch(y, x):
    return float(np.sum(0.5 * (y[:-1] + y[1:]) * np.diff(x)))

def evaluate_financial_impact(y_true, amounts, prob, threshold=0.5):
    y_pred = (prob >= threshold).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    acc = np.mean(y_true == y_pred)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_count = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2.0 * prec * rec_count / (prec + rec_count) if (prec + rec_count) > 0 else 0.0
    
    # Dollar-Weighted Value Recall
    total_fraud_dollars = np.sum(amounts[y_true == 1])
    caught_fraud_dollars = np.sum(amounts[(y_true == 1) & (y_pred == 1)])
    missed_fraud_dollars = np.sum(amounts[(y_true == 1) & (y_pred == 0)])
    rec_dollar = caught_fraud_dollars / total_fraud_dollars if total_fraud_dollars > 0 else 0.0
    
    # ROC-AUC
    th_s = np.linspace(0, 1, 101)
    tprs, fprs = [], []
    P, N = np.sum(y_true == 1), np.sum(y_true == 0)
    for th in th_s:
        yp = (prob >= th).astype(int)
        tprs.append(np.sum((y_true == 1) & (yp == 1)) / P)
        fprs.append(np.sum((y_true == 0) & (yp == 1)) / N)
    fpr_arr = np.array(fpr_list := fprs)
    tpr_arr = np.array(tpr_list := tprs)
    sort_i = np.argsort(fpr_arr)
    auc = trapezoid_scratch(tpr_arr[sort_i], fpr_arr[sort_i])
    
    return {
        'Accuracy': acc,
        'Precision': prec,
        'Count_Recall': rec_count,
        'Dollar_Value_Recall': rec_dollar,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'Dollars_Protected': caught_fraud_dollars,
        'Dollars_Lost': missed_fraud_dollars,
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn)
    }

res_std = evaluate_financial_impact(y_test, amounts_test, standard_model.predict_proba(X_te_norm))
res_cost = evaluate_financial_impact(y_test, amounts_test, cost_model.predict_proba(X_te_norm))

df_comp = pd.DataFrame([res_std, res_cost], index=['Standard Objective (Unweighted)', 'Cost-Sensitive Objective (kappa=2.0)'])
print("\n--- FINANCIAL RECALL & DOLLAR PROTECTION BENCHMARK ---")
df_comp[['Count_Recall', 'Dollar_Value_Recall', 'Dollars_Protected', 'Dollars_Lost', 'ROC-AUC', 'FP', 'FN']]

### 5.1 Visualizing Dollars Protected and Loss Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Dollar Protection Bar Comparison
bars = axes[0].bar(['Standard Model', 'Cost-Sensitive Model'],
                   [res_std['Dollars_Protected'], res_cost['Dollars_Protected']],
                   color=['#1f77b4', '#2ca02c'], width=0.4, edgecolor='black')
axes[0].set_ylabel('Total Stolen Dollars Recovered ($)')
axes[0].set_title('Financial Dollar Capital Protected on Held-Out Test Set')
for bar in bars:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 100, f'${yval:,.2f}', ha='center', va='bottom', fontweight='bold')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# 2. Value Recall vs Count Recall
x = np.arange(2)
w = 0.35
axes[1].bar(x - w/2, [res_std['Count_Recall']*100, res_cost['Count_Recall']*100], w, label='Count Recall (% of events)', color='#1f77b4', alpha=0.85)
axes[1].bar(x + w/2, [res_std['Dollar_Value_Recall']*100, res_cost['Dollar_Value_Recall']*100], w, label='Dollar Value Recall (% of money)', color='#ff7f0e', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Standard Model', 'Cost-Sensitive Model'])
axes[1].set_ylabel('Recall Rate (%)')
axes[1].set_title('Count Recall vs. Dollar Value Recall')
axes[1].set_ylim(80, 102)
axes[1].legend(loc='lower right')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Academic Takeaways & Methodological Insights

1. **Mathematical Impact on Gradient Surfaces:**
   By scaling individual gradient updates by $c^{(i)} = 1 + \kappa \ln(\text{Amount}^{(i)} + 1)$, parameter optimization creates steep gradient descents specifically on high-value fraud instances, ensuring the model's decision boundary never sacrifices high-dollar attacks.
2. **Real-World Banking Alignment:**
   Evaluating models using **Dollar-Weighted Value Recall** ($\text{Recall}_{\$}$) reflects the genuine business goal of credit card issuing banks: maximizing total capital preservation rather than treating all transactions symmetrically.
3. **Zero Package Dependencies:**
   The cost weighting, logarithmic scaling, weighted cross-entropy summation, and analytical gradient updates were implemented 100% from first principles in pure NumPy.